In [2]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

REFERENCE_DATE = pd.Timestamp('2026-01-01')
RANDOM_STATE   = 18

In [3]:
df_train_raw = pd.read_parquet('../Hackathon2026Task-20260328T120731Z-3-001/Hackathon2026Task/block1_train.parquet')  
df_test_raw = pd.read_parquet('../Hackathon2026Task-20260328T120731Z-3-001/Hackathon2026Task/block2_test.parquet')

In [4]:
# Verify claim_free_years has negatives
print("claim_free_years range:")
print(df_train_raw['claim_free_years'].value_counts().sort_index().head(20))

# Verify second driver columns
print("\nSecond driver nulls:")
print(df_train_raw[['second_driver_birthdate', 
                     'second_driver_claim_free_years']].isnull().sum())

claim_free_years range:
claim_free_years
-1      2767
-2      2287
-3      1038
0     159429
1      50057
10     18819
11      8517
12      9723
13      7328
14      6975
15     11817
16      5427
17      5144
18      6130
19      3875
2      41939
20      9845
21      2652
22      3240
23      2708
Name: count, dtype: int64

Second driver nulls:
second_driver_birthdate           541292
second_driver_claim_free_years    541292
dtype: int64


In [6]:
def preprocess(df_input, reference_date=REFERENCE_DATE, 
               is_train=True, maker_encoding=None):
    df = df_input.copy()

    # ── 1. Drop IDs and 100% null ─────────────────────────────────────────
    df.drop(columns=[c for c in [
        'quote_id', 'vehicle_number_plate',
        'second_driver_birthdate',
        'second_driver_claim_free_years',
        'postal_code_houses_owned_by_rental_association_ratio',
    ] if c in df.columns], inplace=True)

    # ── 2. Drop zero variance ─────────────────────────────────────────────
    df.drop(columns=[c for c in [
        'usage', 'vehicle_is_taxi', 'vehicle_number_of_wheels',
        'is_driver_owner', 'vehicle_can_be_registered',
        'vehicle_is_marked_for_export', 'vehicle_model',
        'municipality', 'postal_code', 'province',
        'postal_code_houses_built_before_1945_ratio',
        'postal_code_houses_built_45_to_65_ratio',
        'postal_code_houses_built_65_to_75_ratio',
        'postal_code_houses_built_75_to_85_ratio',
        'postal_code_houses_built_85_to_95_ratio',
        'postal_code_houses_built_95_to_05_ratio',
        'postal_code_houses_built_05_to_15_ratio',
    ] if c in df.columns], inplace=True)

    # ── 3. Dates → numeric ────────────────────────────────────────────────
    if 'contractor_birthdate' in df.columns:
        df['contractor_birthdate'] = pd.to_datetime(
            df['contractor_birthdate'], dayfirst=True, errors='coerce')
        df['contractor_age'] = (
            (reference_date - df['contractor_birthdate']).dt.days // 365)
        df.drop(columns=['contractor_birthdate'], inplace=True)

    date_cols = [
        'vehicle_first_registration_date',
        'vehicle_country_first_registration_date',
        'vehicle_last_registration_date',
        'vehicle_inspection_report_date',
        'vehicle_inspection_expiry_date',
    ]
    for col in [c for c in date_cols if c in df.columns]:
        df[col] = pd.to_datetime(df[col], dayfirst=True, errors='coerce')

    if 'vehicle_first_registration_date' in df.columns:
        df['vehicle_years_since_first_registration'] = (
            (reference_date - df['vehicle_first_registration_date']
             ).dt.days // 365)

    if 'vehicle_inspection_expiry_date' in df.columns:
        df['vehicle_days_until_inspection_expiry'] = (
            df['vehicle_inspection_expiry_date'] - reference_date
        ).dt.days.clip(lower=-3650)

    df.drop(columns=[c for c in date_cols if c in df.columns], inplace=True)

    # ── 4. Convert numeric strings → float ───────────────────────────────
    to_numeric = [
        'vehicle_engine_size', 'vehicle_power', 'vehicle_net_weight',
        'vehicle_gross_weight', 'vehicle_length', 'vehicle_width',
        'vehicle_height', 'vehicle_net_max_power',
        'vehicle_net_max_power_electric', 'vehicle_nominal_continuous_max_power',
        'vehicle_power_to_net_weight_ratio', 'vehicle_number_of_cylinders',
        'vehicle_number_of_doors', 'vehicle_number_of_seats',
        'vehicle_value_new', 'vehicle_planned_annual_mileage',
        'vehicle_age', 'vehicle_years_since_country_first_registration',
        'vehicle_year_of_last_odometer_report',
        'vehicle_inspection_number_of_deficiencies_found',
        'vehicle_ownership_duration', 'claim_free_years',
        'postal_code_urban_category', 'postal_code_latitude',
        'postal_code_longitude', 'postal_code_distance_to_border',
        'postal_code_population', 'postal_code_households',
        'postal_code_houses', 'postal_code_average_household_size',
        'postal_code_average_property_value', 'postal_code_address_density',
        'municipality_crimes_per_1000',
        'postal_code_0_to_15_years_inhabitants_ratio',
        'postal_code_25_to_45_years_inhabitants_ratio',
        'postal_code_45_to_65_years_inhabitants_ratio',
        'postal_code_65_years_and_older_inhabitants_ratio',
        'postal_code_single_person_households_ratio',
        'postal_code_multi_person_households_without_children_ratio',
        'postal_code_two_parent_households_ratio',
        'postal_code_social_benefit_recipients_ratio',
        'postal_code_owner_occupied_houses_ratio',
        'postal_code_rental_houses_ratio',
        'postal_code_multi_family_houses_ratio',
        'postal_code_nearest_hospital_excl_clinic_distance',
        'postal_code_nearest_motorway_ramp_distance',
        'postal_code_nearest_train_station_distance',
        'postal_code_hospitals_excl_clinic_within_10_km',
        'postal_code_supermarkets_within_3_km',
        'postal_code_restaurants_within_3_km',
        'postal_code_general_practitioners_within_3_km',
        'postal_code_nearest_general_practitioner_distance',
    ]
    for col in [c for c in to_numeric if c in df.columns]:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    # ── 5. Drop redundant distance/within cols ────────────────────────────
    keep_distance = [
        'postal_code_nearest_hospital_excl_clinic_distance',
        'postal_code_nearest_motorway_ramp_distance',
        'postal_code_nearest_train_station_distance',
        'postal_code_nearest_general_practitioner_distance',
    ]
    keep_within = [
        'postal_code_hospitals_excl_clinic_within_10_km',
        'postal_code_supermarkets_within_3_km',
        'postal_code_restaurants_within_3_km',
        'postal_code_general_practitioners_within_3_km',
    ]
    drop_dist  = [c for c in df.columns if 'nearest' in c 
                  and c not in keep_distance]
    drop_with  = [c for c in df.columns if 'within' in c 
                  and c not in keep_within]
    df.drop(columns=[c for c in drop_dist + drop_with 
                     if c in df.columns], inplace=True)

    # ── 6. Electric vehicle ───────────────────────────────────────────────
    df = df.copy()
    if 'vehicle_net_max_power_electric' in df.columns:
        df['vehicle_is_electric'] = (
            df['vehicle_net_max_power_electric'].notnull().astype(int))
        df['vehicle_net_max_power_electric'] = (
            df['vehicle_net_max_power_electric'].fillna(0))
    if 'vehicle_nominal_continuous_max_power' in df.columns:
        df['vehicle_nominal_continuous_max_power'] = (
            df['vehicle_nominal_continuous_max_power'].fillna(0))

    # ── 7. Binary yes/no → 0/1 ───────────────────────────────────────────
    for col in ['vehicle_is_imported',
                'vehicle_is_imported_within_last_12_months',
                'vehicle_has_open_recall']:
        if col in df.columns and df[col].dtype == object:
            df[col] = df[col].map({'yes': 1, 'no': 0})
    if 'vehicle_is_imported_within_last_12_months' in df.columns:
        df['vehicle_is_imported_within_last_12_months'] = (
            df['vehicle_is_imported_within_last_12_months'].fillna(0).astype(int))

    # ── 8. Payment frequency ─────────────────────────────────────────────
    if 'payment_frequency' in df.columns and df['payment_frequency'].dtype == object:
        df['payment_frequency'] = df['payment_frequency'].map(
            {'monthly': 0, 'yearly': 1})

    # ── 9. Claim free years engineering ──────────────────────────────────
    if 'claim_free_years' in df.columns:
        df['claim_free_years'] = pd.to_numeric(
            df['claim_free_years'], errors='coerce')
        df['has_recent_claim']  = (df['claim_free_years'] < 0).astype(int)
        df['claim_severity']    = df['claim_free_years'].clip(upper=0).abs()
        df['is_experienced']    = (df['claim_free_years'] > 10).astype(int)
        df['is_elderly_driver'] = (df['claim_free_years'] > 30).astype(int)

    # ── 10. Feature engineering ───────────────────────────────────────────
    df = df.copy()
    if 'vehicle_power' in df.columns and 'vehicle_value_new' in df.columns:
        df['power_per_value']  = (
            df['vehicle_power'] / (df['vehicle_value_new'] + 1))
    if 'vehicle_value_new' in df.columns and 'vehicle_age' in df.columns:
        df['value_per_age']    = (
            df['vehicle_value_new'] / (df['vehicle_age'] + 1))
    if 'vehicle_age' in df.columns and 'vehicle_power' in df.columns:
        df['age_times_power']  = df['vehicle_age'] * df['vehicle_power']
    if 'contractor_age' in df.columns:
        df['contractor_age_sq'] = df['contractor_age'] ** 2
    if 'claim_free_years' in df.columns and 'contractor_age' in df.columns:
        df['age_x_claimfree']  = (
            df['contractor_age'] * df['claim_free_years'])

    # ── 11. One-hot encode ────────────────────────────────────────────────
    onehot_cols = ['vehicle_fuel_type', 'vehicle_primary_color',
                   'vehicle_odometer_verdict_code']
    for col in [c for c in onehot_cols if c in df.columns]:
        dummies = pd.get_dummies(df[col], prefix=col, drop_first=False)
        df = pd.concat([df, dummies], axis=1)
        df.drop(columns=[col], inplace=True)

    # ── 12. Coverage encoding ─────────────────────────────────────────────
    if 'coverage' in df.columns and df['coverage'].dtype == object:
        df['coverage'] = df['coverage'].map(
            {'mtpl': 0, 'limited_casco': 1, 'casco': 2})

    # ── 13. Target encode vehicle_maker ──────────────────────────────────
    df = df.copy()
    if 'vehicle_maker' in df.columns:
        if is_train:
            price_cols     = [c for c in df.columns if c.endswith('_price')]
            maker_encoding = (df.groupby('vehicle_maker')[price_cols]
                              .mean().mean(axis=1).to_dict())
        global_mean = sum(maker_encoding.values()) / len(maker_encoding)
        df['vehicle_maker_encoded'] = (
            df['vehicle_maker'].map(maker_encoding).fillna(global_mean))
        df.drop(columns=['vehicle_maker'], inplace=True)

    # ── 14. Add insurer quoted flags (train only) ─────────────────────────
    if is_train:
        for col in [c for c in df.columns if c.endswith('_price')]:
            insurer = col.replace('_price', '')
            df[f'{insurer}_quoted'] = df[col].notnull().astype(int)

    df = df.copy()
    return (df, maker_encoding) if is_train else df

In [7]:
df_train, maker_enc = preprocess(df_train_raw, is_train=True)
df_test             = preprocess(df_test_raw,  is_train=False, 
                                  maker_encoding=maker_enc)

print(f"Train shape: {df_train.shape}")
print(f"Test shape:  {df_test.shape}")
print(f"Object cols: {df_train.select_dtypes(include='object').columns.tolist()}")

# Verify claim engineering
print("\nNew claim features:")
print(df_train[['claim_free_years', 'has_recent_claim', 
                 'claim_severity', 'is_experienced', 
                 'is_elderly_driver']].describe())

Train shape: (541292, 133)
Test shape:  (164092, 111)
Object cols: []

New claim features:
       claim_free_years  has_recent_claim  claim_severity  is_experienced  \
count     541292.000000     541292.000000   541292.000000   541292.000000   
mean           6.234716          0.011255        0.019315        0.211187   
std            8.350639          0.105489        0.197225        0.408151   
min           -3.000000          0.000000        0.000000        0.000000   
25%            0.000000          0.000000        0.000000        0.000000   
50%            3.000000          0.000000        0.000000        0.000000   
75%            9.000000          0.000000        0.000000        0.000000   
max           82.000000          1.000000        3.000000        1.000000   

       is_elderly_driver  
count      541292.000000  
mean            0.018399  
std             0.134388  
min             0.000000  
25%             0.000000  
50%             0.000000  
75%             0.000000  

In [8]:
insurer_price_cols = [c for c in df_train_raw.columns if c.endswith('_price')]

print("=== PER INSURER STATISTICS ===\n")
for insurer in insurer_price_cols:
    vals = df_train_raw[insurer].dropna()
    print(f"{insurer}:")
    print(f"  quote_rate={len(vals)/len(df_train_raw)*100:.1f}%  "
          f"n={len(vals):,}  "
          f"mean={vals.mean():.1f}  "
          f"median={vals.median():.1f}  "
          f"std={vals.std():.1f}  "
          f"skew={vals.skew():.2f}  "
          f"min={vals.min():.1f}  "
          f"max={vals.max():.1f}")

=== PER INSURER STATISTICS ===

Insurer_A_price:
  quote_rate=97.4%  n=527,372  mean=96.9  median=67.8  std=115.5  skew=8.89  min=12.3  max=4370.4
Insurer_B_price:
  quote_rate=80.5%  n=435,759  mean=99.8  median=76.3  std=75.5  skew=2.51  min=15.8  max=1252.5
Insurer_C_price:
  quote_rate=79.6%  n=431,011  mean=89.3  median=70.8  std=62.9  skew=2.40  min=12.2  max=1095.2
Insurer_D_price:
  quote_rate=74.5%  n=403,079  mean=98.5  median=75.8  std=71.3  skew=2.12  min=14.0  max=858.6
Insurer_E_price:
  quote_rate=74.4%  n=402,556  mean=115.7  median=90.3  std=88.2  skew=3.26  min=15.6  max=2283.8
Insurer_F_price:
  quote_rate=66.1%  n=357,844  mean=97.4  median=79.1  std=66.1  skew=2.08  min=14.1  max=781.4
Insurer_G_price:
  quote_rate=58.1%  n=314,221  mean=84.5  median=70.3  std=51.6  skew=1.73  min=12.5  max=646.5
Insurer_H_price:
  quote_rate=21.0%  n=113,403  mean=116.1  median=92.4  std=76.0  skew=1.82  min=15.2  max=996.0
Insurer_I_price:
  quote_rate=53.5%  n=289,429  mean=94.9

In [8]:
print("=== MEAN PRICE BY COVERAGE ===\n")
for insurer in insurer_price_cols:
    print(f"{insurer}:")
    for cov in ['mtpl', 'limited_casco', 'casco']:
        vals = df_train_raw[df_train_raw['coverage'] == cov][insurer].dropna()
        print(f"  {cov:15s} mean={vals.mean():7.1f}  "
              f"median={vals.median():7.1f}  "
              f"std={vals.std():6.1f}  "
              f"n={len(vals):,}")

=== MEAN PRICE BY COVERAGE ===

Insurer_A_price:
  mtpl            mean=   86.4  median=   65.2  std=  87.3  n=139,374
  limited_casco   mean=   92.4  median=   63.2  std= 100.9  n=231,334
  casco           mean=  112.9  median=   76.5  std= 150.8  n=156,664
Insurer_B_price:
  mtpl            mean=   98.3  median=   77.8  std=  69.7  n=108,189
  limited_casco   mean=   88.0  median=   67.0  std=  69.3  n=186,607
  casco           mean=  116.5  median=   88.1  std=  84.2  n=140,963
Insurer_C_price:
  mtpl            mean=   70.6  median=   63.0  std=  34.1  n=98,257
  limited_casco   mean=   82.2  median=   64.0  std=  60.6  n=192,485
  casco           mean=  112.2  median=   89.5  std=  73.8  n=140,269
Insurer_D_price:
  mtpl            mean=   93.9  median=   71.7  std=  66.9  n=109,973
  limited_casco   mean=   90.1  median=   69.4  std=  69.2  n=158,452
  casco           mean=  112.1  median=   88.5  std=  75.2  n=134,654
Insurer_E_price:
  mtpl            mean=  124.0  median=   98

In [9]:
print("=== MEAN PRICE BY CLAIM HISTORY ===\n")
df_train_raw['claim_free_years_num'] = pd.to_numeric(
    df_train_raw['claim_free_years'], errors='coerce')

# Segment: bad driver, new driver, experienced, very experienced
df_train_raw['driver_segment'] = pd.cut(
    df_train_raw['claim_free_years_num'],
    bins=[-999, -1, 0, 5, 10, 30, 999],
    labels=['bad_driver', 'zero_cfr', 'new_driver', 
            'moderate', 'experienced', 'very_experienced']
)

for insurer in insurer_price_cols:
    print(f"\n{insurer}:")
    print(df_train_raw.groupby('driver_segment')[insurer]
          .agg(['mean', 'median', 'count'])
          .round(1).to_string())

=== MEAN PRICE BY CLAIM HISTORY ===


Insurer_A_price:
                   mean  median   count
driver_segment                         
bad_driver        121.7   101.9    6069
zero_cfr          133.2   106.9  157947
new_driver         94.1    70.0  178995
moderate           66.0    46.2   76227
experienced        68.8    43.0   98928
very_experienced   69.1    40.0    9206

Insurer_B_price:
                   mean  median   count
driver_segment                         
bad_driver        204.6   174.8     967
zero_cfr          156.1   128.3  121129
new_driver         99.9    80.8  146961
moderate           59.7    49.3   69168
experienced        57.1    52.0   89084
very_experienced   56.9    53.9    8450

Insurer_C_price:
                   mean  median   count
driver_segment                         
bad_driver        145.0   123.8    4703
zero_cfr          126.0   103.1  107865
new_driver         94.7    77.8  149754
moderate           63.3    53.0   70473
experienced        57.2    50

In [10]:
print("=== MEAN PRICE BY VEHICLE VALUE SEGMENT ===\n")
df_train_raw['vehicle_value_new_num'] = pd.to_numeric(
    df_train_raw['vehicle_value_new'], errors='coerce')

df_train_raw['value_segment'] = pd.qcut(
    df_train_raw['vehicle_value_new_num'], 
    q=4, 
    labels=['budget', 'mid', 'premium', 'luxury']
)

for insurer in insurer_price_cols:
    print(f"\n{insurer}:")
    print(df_train_raw.groupby('value_segment')[insurer]
          .agg(['mean', 'median', 'count'])
          .round(1).to_string())

=== MEAN PRICE BY VEHICLE VALUE SEGMENT ===


Insurer_A_price:
                mean  median   count
value_segment                       
budget          78.5    56.5  132311
mid             83.9    59.6  131147
premium         95.9    66.8  129961
luxury         129.2    93.5  131224

Insurer_B_price:
                mean  median   count
value_segment                       
budget          92.8    70.0  122636
mid             91.2    69.8  115808
premium         95.3    72.5  108833
luxury         126.3    98.5   87984

Insurer_C_price:
                mean  median   count
value_segment                       
budget          65.8    54.4  105519
mid             79.3    63.1  111669
premium         93.5    73.4  110986
luxury         120.3    99.2  101727

Insurer_D_price:
                mean  median   count
value_segment                       
budget          87.6    65.2  104816
mid             94.2    70.7  106106
premium         98.7    74.1  101936
luxury         116.2    95.8   8

In [11]:
print("=== DEDUCTIBLE ANALYSIS ===\n")
deductible_cols = [c for c in df_train_raw.columns if c.endswith('_deductible')]

for ded in deductible_cols:
    vals = df_train_raw[ded].dropna()
    insurer = ded.replace('_deductible', '')
    price   = insurer + '_price'
    
    print(f"{ded}:")
    print(f"  unique values: {sorted(vals.unique())}")
    
    # Correlation between deductible and price
    if price in df_train_raw.columns:
        corr = df_train_raw[[ded, price]].dropna().corr().iloc[0,1]
        print(f"  corr with price: {corr:.3f}")
    print()

=== DEDUCTIBLE ANALYSIS ===

Insurer_A_deductible:
  unique values: [0.0, 150.0, 250.0, 500.0, 1000.0, 2500.0]
  corr with price: 0.103

Insurer_B_deductible:
  unique values: [0.0, 150.0, 250.0, 500.0, 1000.0]
  corr with price: 0.017

Insurer_C_deductible:
  unique values: [0.0, 135.0]
  corr with price: 0.162

Insurer_D_deductible:
  unique values: [0.0, 250.0, 500.0, 1000.0, 2500.0]
  corr with price: 0.004

Insurer_E_deductible:
  unique values: [0.0, 135.0, 150.0, 250.0, 300.0, 500.0, 600.0, 750.0, 1000.0, 1200.0]
  corr with price: 0.026

Insurer_F_deductible:
  unique values: [0.0, 250.0, 500.0, 750.0, 1000.0]
  corr with price: 0.010

Insurer_G_deductible:
  unique values: [0.0, 250.0, 500.0, 1000.0]
  corr with price: 0.076

Insurer_H_deductible:
  unique values: [0.0, 135.0, 150.0, 250.0, 300.0, 500.0, 600.0, 1000.0, 1200.0]
  corr with price: 0.112

Insurer_I_deductible:
  unique values: [0.0, 250.0, 500.0, 1000.0, 2500.0]
  corr with price: -0.003

Insurer_J_deductible:
  

In [12]:
print("=== LOCATION & DEMOGRAPHICS SIGNAL ===\n")

# Convert key columns
location_cols = [
    'postal_code_urban_category',
    'postal_code_average_property_value', 
    'municipality_crimes_per_1000',
    'postal_code_population',
    'postal_code_address_density',
    'postal_code_0_to_15_years_inhabitants_ratio',
    'postal_code_65_years_and_older_inhabitants_ratio',
    'postal_code_single_person_households_ratio',
    'postal_code_social_benefit_recipients_ratio',
    'postal_code_distance_to_border',
    'postal_code_latitude',
    'postal_code_longitude',
]

df_check = df_train_raw.copy()
for col in location_cols:
    df_check[col] = pd.to_numeric(df_check[col], errors='coerce')

print(f"{'Feature':50s} {'corr_avg':>10s}", end="")
for ins in ['A','E','G','H','I']:
    print(f"  {'Ins_'+ins:>8s}", end="")
print()
print("-" * 100)

for col in location_cols:
    corrs = []
    row = f"  {col:48s}"
    for insurer in insurer_price_cols:
        c = df_check[[col, insurer]].dropna().corr().iloc[0,1]
        corrs.append(c)
    avg = np.mean([abs(c) for c in corrs])
    row += f" {avg:10.3f}"
    for ins in ['A','E','G','H','I']:
        c = df_check[[col, f'Insurer_{ins}_price']].dropna().corr().iloc[0,1]
        row += f"  {c:8.3f}"
    print(row)

=== LOCATION & DEMOGRAPHICS SIGNAL ===

Feature                                              corr_avg     Ins_A     Ins_E     Ins_G     Ins_H     Ins_I
----------------------------------------------------------------------------------------------------
  postal_code_urban_category                            0.194    -0.082    -0.147    -0.281    -0.222    -0.281
  postal_code_average_property_value                    0.054     0.027    -0.055    -0.039     0.058    -0.134
  municipality_crimes_per_1000                          0.048     0.019     0.061     0.049     0.070     0.055
  postal_code_population                                0.043     0.022     0.040     0.052     0.082     0.044
  postal_code_address_density                           0.144     0.063     0.119     0.191     0.173     0.207
  postal_code_0_to_15_years_inhabitants_ratio           0.023     0.023     0.056     0.011    -0.004     0.003
  postal_code_65_years_and_older_inhabitants_ratio      0.069    -0.039    

In [13]:
print("=== VEHICLE PHYSICAL ATTRIBUTES SIGNAL ===\n")

vehicle_cols = [
    'vehicle_value_new',
    'vehicle_engine_size',
    'vehicle_power',
    'vehicle_net_weight',
    'vehicle_gross_weight',
    'vehicle_length',
    'vehicle_width', 
    'vehicle_height',
    'vehicle_number_of_doors',
    'vehicle_number_of_seats',
    'vehicle_number_of_cylinders',
    'vehicle_planned_annual_mileage',
    'vehicle_net_max_power',
    'vehicle_power_to_net_weight_ratio',
]

df_check2 = df_train_raw.copy()
for col in vehicle_cols:
    df_check2[col] = pd.to_numeric(df_check2[col], errors='coerce')

print(f"{'Feature':45s} {'avg_abs_corr':>12s}", end="")
for ins in ['A','E','G','H','I']:
    print(f"  {'Ins_'+ins:>8s}", end="")
print()
print("-" * 95)

for col in vehicle_cols:
    corrs = []
    row = f"  {col:43s}"
    for insurer in insurer_price_cols:
        c = df_check2[[col, insurer]].dropna().corr().iloc[0,1]
        corrs.append(c)
    avg = np.mean([abs(c) for c in corrs])
    row += f" {avg:12.3f}"
    for ins in ['A','E','G','H','I']:
        c = df_check2[[col, f'Insurer_{ins}_price']].dropna().corr().iloc[0,1]
        row += f"  {c:8.3f}"
    print(row)

=== VEHICLE PHYSICAL ATTRIBUTES SIGNAL ===

Feature                                       avg_abs_corr     Ins_A     Ins_E     Ins_G     Ins_H     Ins_I
-----------------------------------------------------------------------------------------------
  vehicle_value_new                                  0.194     0.183     0.141     0.230     0.239     0.058
  vehicle_engine_size                                0.118     0.156     0.099     0.109     0.139     0.044
  vehicle_power                                      0.176     0.183     0.127     0.187     0.242     0.028
  vehicle_net_weight                                 0.189     0.178     0.158     0.230     0.275     0.058
  vehicle_gross_weight                               0.172     0.165     0.139     0.205     0.261     0.043
  vehicle_length                                     0.059     0.046     0.021     0.087     0.111    -0.025
  vehicle_width                                      0.055     0.032     0.020     0.082     0.12

In [14]:
print("=== DRIVER PROFILE SIGNAL ===\n")

df_check3 = df_train_raw.copy()
df_check3['contractor_age'] = (
    pd.Timestamp('2025-01-01') - 
    pd.to_datetime(df_check3['contractor_birthdate'], 
                   dayfirst=True, errors='coerce')
).dt.days // 365

df_check3['claim_free_years_num'] = pd.to_numeric(
    df_check3['claim_free_years'], errors='coerce')

df_check3['vehicle_ownership_duration_num'] = pd.to_numeric(
    df_check3['vehicle_ownership_duration'], errors='coerce')

driver_cols = [
    'contractor_age',
    'claim_free_years_num',
    'vehicle_ownership_duration_num',
    'vehicle_planned_annual_mileage',
]

print(f"{'Feature':45s} {'avg_abs_corr':>12s}", end="")
for ins in ['A','E','G','H','I']:
    print(f"  {'Ins_'+ins:>8s}", end="")
print()
print("-" * 95)

for col in driver_cols:
    df_check3[col] = pd.to_numeric(df_check3[col], errors='coerce')
    row = f"  {col:43s}"
    corrs = []
    for insurer in insurer_price_cols:
        c = df_check3[[col, insurer]].dropna().corr().iloc[0,1]
        corrs.append(c)
    avg = np.mean([abs(c) for c in corrs])
    row += f" {avg:12.3f}"
    for ins in ['A','E','G','H','I']:
        c = df_check3[[col, f'Insurer_{ins}_price']].dropna().corr().iloc[0,1]
        row += f"  {c:8.3f}"
    print(row)

=== DRIVER PROFILE SIGNAL ===

Feature                                       avg_abs_corr     Ins_A     Ins_E     Ins_G     Ins_H     Ins_I
-----------------------------------------------------------------------------------------------
  contractor_age                                     0.344    -0.185    -0.339    -0.311    -0.208    -0.317
  claim_free_years_num                               0.363    -0.169    -0.334    -0.370    -0.356    -0.351
  vehicle_ownership_duration_num                     0.194    -0.104    -0.186    -0.216    -0.169    -0.166
  vehicle_planned_annual_mileage                     0.046     0.052     0.041     0.024     0.046    -0.015


In [15]:
print("=== VEHICLE ADMINISTRATIVE SIGNAL ===\n")

df_check4 = df_train_raw.copy()
ref = pd.Timestamp('2025-01-01')

df_check4['vehicle_first_registration_date'] = pd.to_datetime(
    df_check4['vehicle_first_registration_date'], 
    dayfirst=True, errors='coerce')
df_check4['vehicle_years_since_first_reg'] = (
    (ref - df_check4['vehicle_first_registration_date']).dt.days // 365)

df_check4['vehicle_inspection_expiry_date'] = pd.to_datetime(
    df_check4['vehicle_inspection_expiry_date'],
    dayfirst=True, errors='coerce')
df_check4['days_until_inspection_expiry'] = (
    df_check4['vehicle_inspection_expiry_date'] - ref).dt.days

admin_cols = [
    'vehicle_age',
    'vehicle_years_since_first_reg',
    'vehicle_year_of_last_odometer_report',
    'vehicle_inspection_number_of_deficiencies_found',
    'days_until_inspection_expiry',
    'vehicle_years_since_country_first_registration',
]

for col in admin_cols:
    df_check4[col] = pd.to_numeric(df_check4[col], errors='coerce')

print(f"{'Feature':50s} {'avg_abs_corr':>12s}", end="")
for ins in ['A','E','G','H','I']:
    print(f"  {'Ins_'+ins:>8s}", end="")
print()
print("-" * 100)

for col in admin_cols:
    row = f"  {col:48s}"
    corrs = []
    for insurer in insurer_price_cols:
        c = df_check4[[col, insurer]].dropna().corr().iloc[0,1]
        corrs.append(c)
    avg = np.mean([abs(c) for c in corrs])
    row += f" {avg:12.3f}"
    for ins in ['A','E','G','H','I']:
        c = df_check4[[col, f'Insurer_{ins}_price']].dropna().corr().iloc[0,1]
        row += f"  {c:8.3f}"
    print(row)

=== VEHICLE ADMINISTRATIVE SIGNAL ===

Feature                                            avg_abs_corr     Ins_A     Ins_E     Ins_G     Ins_H     Ins_I
----------------------------------------------------------------------------------------------------
  vehicle_age                                             0.098    -0.051     0.014    -0.150    -0.192     0.084
  vehicle_years_since_first_reg                           0.097    -0.051     0.015    -0.147    -0.191     0.085
  vehicle_year_of_last_odometer_report                    0.010    -0.007    -0.000    -0.015     0.002     0.017
  vehicle_inspection_number_of_deficiencies_found           nan       nan       nan       nan       nan       nan
  days_until_inspection_expiry                            0.041     0.026     0.026     0.055     0.036     0.008
  vehicle_years_since_country_first_registration          0.094    -0.063    -0.003    -0.130    -0.167     0.070


In [16]:
print("=== CATEGORICAL FEATURES SIGNAL ===\n")

cat_cols = [
    'coverage',
    'vehicle_fuel_type', 
    'vehicle_primary_color',
    'vehicle_maker',
    'vehicle_odometer_verdict_code',
    'payment_frequency',
    'vehicle_is_imported',
    'vehicle_has_open_recall',
]

for col in cat_cols:
    print(f"\n{col}:")
    # Mean price per category for key insurers
    for ins in ['A', 'E', 'G', 'H']:
        insurer = f'Insurer_{ins}_price'
        means = df_train_raw.groupby(col)[insurer].mean().round(1)
        std   = means.std().round(1)
        print(f"  Insurer_{ins} — std across categories: {std:.1f}  "
              f"values: {means.to_dict()}")

=== CATEGORICAL FEATURES SIGNAL ===


coverage:
  Insurer_A — std across categories: 13.9  values: {'casco': 112.9, 'limited_casco': 92.4, 'mtpl': 86.4}
  Insurer_E — std across categories: 14.4  values: {'casco': 129.1, 'limited_casco': 102.0, 'mtpl': 124.0}
  Insurer_G — std across categories: 14.4  values: {'casco': 101.6, 'limited_casco': 74.9, 'mtpl': 78.7}
  Insurer_H — std across categories: 30.3  values: {'casco': 148.2, 'limited_casco': 142.7, 'mtpl': 93.1}

vehicle_fuel_type:
  Insurer_A — std across categories: 18.2  values: {'diesel': 109.8, 'electric': 135.7, 'gas': 108.9, 'other': 142.2, 'petrol': 87.4, 'petrol_electric_hybrid': 120.7, 'petrol_gas': 115.8}
  Insurer_E — std across categories: 21.9  values: {'diesel': 146.2, 'electric': 158.7, 'gas': 170.9, 'other': 137.1, 'petrol': 107.6, 'petrol_electric_hybrid': 134.1, 'petrol_gas': 118.9}
  Insurer_G — std across categories: 11.9  values: {'diesel': 97.1, 'electric': 109.7, 'gas': 101.1, 'other': 99.7, 'petrol': 77.2, 

In [9]:
# Instead of one global maker encoding, create per-insurer encodings
# This captures that Lucid is €341 for A but different for others
maker_encodings = {}
for insurer in insurer_price_cols:
    maker_encodings[insurer] = (
        df_train_raw.groupby('vehicle_maker')[insurer]
        .mean().to_dict()
    )

In [10]:
# Step 1 — Base preprocessing (already done)
# df_train and df_test are already preprocessed

# Step 2 — Per-insurer feature sets
# Each insurer gets base features + its specific extras

INSURER_FEATURES = {
    'Insurer_A_price': {
        'extra_features': [],  # payment_frequency already in base
        'drop_features':  ['postal_code_social_benefit_recipients_ratio'],
        'maker_encode':   True,   # per-insurer maker encoding
        'log_transform':  True,
        'params': {'num_leaves': 127, 'n_estimators': 5000, 
                   'learning_rate': 0.01}
    },
    'Insurer_E_price': {
        'extra_features': ['is_elderly_driver', 'claim_severity'],
        'drop_features':  [],
        'maker_encode':   True,   # std 68 — most maker sensitive
        'log_transform':  True,
        'segment_col':    'is_elderly_driver',  # train two models
        'params': {'num_leaves': 63, 'n_estimators': 5000,
                   'learning_rate': 0.01}
    },
    'Insurer_H_price': {
        'extra_features': ['vehicle_length', 'vehicle_width',
                           'vehicle_number_of_doors', 'vehicle_number_of_seats'],
        'drop_features':  [],
        'maker_encode':   True,   # std 52
        'log_transform':  True,
        'params': {'num_leaves': 63, 'n_estimators': 5000,
                   'learning_rate': 0.01}
    },
    'Insurer_I_price': {
        'extra_features': ['old_vehicle_flag'],  # older = more expensive for I
        'drop_features':  ['vehicle_power', 'vehicle_gross_weight',
                           'vehicle_net_weight'],  # I ignores physical
        'maker_encode':   False,
        'log_transform':  True,
        'params': {'num_leaves': 127, 'n_estimators': 5000,
                   'learning_rate': 0.01}
    },
    # B, C, D, F, G, J, K — base features work well
    'Insurer_B_price': {
        'extra_features': [], 'drop_features': [],
        'maker_encode': False, 'log_transform': True,
        'params': {'num_leaves': 127, 'n_estimators': 5000,
                   'learning_rate': 0.01}
    },
    'Insurer_C_price': {
        'extra_features': [], 'drop_features': [],
        'maker_encode': False, 'log_transform': True,
        'params': {'num_leaves': 127, 'n_estimators': 5000,
                   'learning_rate': 0.01}
    },
    'Insurer_D_price': {
        'extra_features': [], 'drop_features': [],
        'maker_encode': False, 'log_transform': True,
        'params': {'num_leaves': 127, 'n_estimators': 5000,
                   'learning_rate': 0.01}
    },
    'Insurer_F_price': {
        'extra_features': [], 'drop_features': [],
        'maker_encode': False, 'log_transform': True,
        'params': {'num_leaves': 127, 'n_estimators': 5000,
                   'learning_rate': 0.01}
    },
    'Insurer_G_price': {
        'extra_features': [], 'drop_features': [],
        'maker_encode': False, 'log_transform': True,
        'params': {'num_leaves': 63, 'n_estimators': 5000,
                   'learning_rate': 0.01}
    },
    'Insurer_J_price': {
        'extra_features': [], 'drop_features': [],
        'maker_encode': False, 'log_transform': True,
        'params': {'num_leaves': 127, 'n_estimators': 5000,
                   'learning_rate': 0.01}
    },
    'Insurer_K_price': {
        'extra_features': [], 'drop_features': [],
        'maker_encode': False, 'log_transform': True,
        'params': {'num_leaves': 127, 'n_estimators': 5000,
                   'learning_rate': 0.01}
    },
}

In [11]:
# Add back to df_train and df_test from raw data
cols_to_add_back = [
    'vehicle_length',    # for H
    'vehicle_width',     # for H  
    'vehicle_number_of_doors',  # for H
    'vehicle_number_of_seats',  # for H
    'postal_code_latitude',     # geographic signal
    'postal_code_longitude',    # geographic signal (std 0.122)
    'postal_code_social_benefit_recipients_ratio',  # for I and E
    'municipality_crimes_per_1000',  # location risk
    'vehicle_has_open_recall',  # recall signal
]

for col in cols_to_add_back:
    if col not in df_train.columns:
        df_train[col] = pd.to_numeric(df_train_raw[col], errors='coerce')
        df_test[col]  = pd.to_numeric(df_test_raw[col],  errors='coerce')

# Add old vehicle flag for Insurer I
df_train['old_vehicle_flag'] = (df_train['vehicle_age'] > 15).astype(int)
df_test['old_vehicle_flag']  = (df_test['vehicle_age']  > 15).astype(int)

# Per-insurer maker encodings
maker_encodings = {}
for insurer in [c for c in df_train.columns if c.endswith('_price')]:
    maker_encodings[insurer] = (
        df_train_raw.groupby('vehicle_maker')[insurer]
        .mean().to_dict()
    )
    
print("Added back features and computed per-insurer maker encodings")
print(f"Train shape: {df_train.shape}")

Added back features and computed per-insurer maker encodings
Train shape: (541292, 134)


In [12]:
print(f"Test shape: {df_test.shape}")

# Verify all added columns exist in both
cols_to_verify = [
    'vehicle_length', 'vehicle_width', 'vehicle_number_of_doors',
    'vehicle_number_of_seats', 'vehicle_has_open_recall',
    'postal_code_social_benefit_recipients_ratio',
    'municipality_crimes_per_1000', 'postal_code_latitude',
    'postal_code_longitude', 'old_vehicle_flag'
]

print("\nColumn presence check:")
for col in cols_to_verify:
    in_train = col in df_train.columns
    in_test  = col in df_test.columns
    flag = "" if (in_train and in_test) else " ← MISSING"
    print(f"  {col:45s} train={in_train}  test={in_test}{flag}")

Test shape: (164092, 112)

Column presence check:
  vehicle_length                                train=True  test=True
  vehicle_width                                 train=True  test=True
  vehicle_number_of_doors                       train=True  test=True
  vehicle_number_of_seats                       train=True  test=True
  vehicle_has_open_recall                       train=True  test=True
  postal_code_social_benefit_recipients_ratio   train=True  test=True
  municipality_crimes_per_1000                  train=True  test=True
  postal_code_latitude                          train=True  test=True
  postal_code_longitude                         train=True  test=True
  old_vehicle_flag                              train=True  test=True


In [21]:
import pickle
import os
os.makedirs('saved_models_v2', exist_ok=True)

# Base feature cols — exclude prices, quoted flags, coverage
base_feature_cols = [c for c in df_train.columns
                     if not c.endswith('_price')
                     and not c.endswith('_quoted')
                     and c != 'coverage']

# Per-insurer config
INSURER_CONFIG = {
    'Insurer_A_price': {
        'drop_features':  ['postal_code_social_benefit_recipients_ratio'],
        'maker_encode':   True,
        'log_transform':  True,
        'params': {'num_leaves': 127, 'n_estimators': 5000,
                   'learning_rate': 0.01, 'min_child_samples': 20,
                   'subsample': 0.8, 'colsample_bytree': 0.8,
                   'reg_alpha': 0.1, 'reg_lambda': 0.1},
    },
    'Insurer_B_price': {
        'drop_features':  ['vehicle_length', 'vehicle_width',
                           'vehicle_number_of_doors', 'old_vehicle_flag'],
        'maker_encode':   False,
        'log_transform':  True,
        'params': {'num_leaves': 127, 'n_estimators': 5000,
                   'learning_rate': 0.01, 'min_child_samples': 20,
                   'subsample': 0.8, 'colsample_bytree': 0.8,
                   'reg_alpha': 0.1, 'reg_lambda': 0.1},
    },
    'Insurer_C_price': {
        'drop_features':  ['vehicle_length', 'vehicle_width',
                           'vehicle_number_of_doors', 'old_vehicle_flag'],
        'maker_encode':   False,
        'log_transform':  True,
        'params': {'num_leaves': 127, 'n_estimators': 5000,
                   'learning_rate': 0.01, 'min_child_samples': 20,
                   'subsample': 0.8, 'colsample_bytree': 0.8,
                   'reg_alpha': 0.1, 'reg_lambda': 0.1},
    },
    'Insurer_D_price': {
        'drop_features':  ['vehicle_length', 'vehicle_width',
                           'old_vehicle_flag'],
        'maker_encode':   False,
        'log_transform':  True,
        'params': {'num_leaves': 127, 'n_estimators': 5000,
                   'learning_rate': 0.01, 'min_child_samples': 20,
                   'subsample': 0.8, 'colsample_bytree': 0.8,
                   'reg_alpha': 0.1, 'reg_lambda': 0.1},
    },
    'Insurer_E_price': {
        'drop_features':  ['vehicle_length', 'vehicle_width',
                           'vehicle_number_of_doors'],
        'maker_encode':   True,
        'log_transform':  True,
        'params': {'num_leaves': 63, 'n_estimators': 5000,
                   'learning_rate': 0.01, 'min_child_samples': 50,
                   'subsample': 0.7, 'colsample_bytree': 0.7,
                   'reg_alpha': 0.3, 'reg_lambda': 0.3},
    },
    'Insurer_F_price': {
        'drop_features':  ['vehicle_length', 'vehicle_width',
                           'vehicle_number_of_doors', 'old_vehicle_flag'],
        'maker_encode':   False,
        'log_transform':  True,
        'params': {'num_leaves': 127, 'n_estimators': 5000,
                   'learning_rate': 0.01, 'min_child_samples': 20,
                   'subsample': 0.8, 'colsample_bytree': 0.8,
                   'reg_alpha': 0.1, 'reg_lambda': 0.1},
    },
    'Insurer_G_price': {
        'drop_features':  ['vehicle_length', 'vehicle_width',
                           'vehicle_number_of_doors', 'old_vehicle_flag'],
        'maker_encode':   False,
        'log_transform':  True,
        'params': {'num_leaves': 63, 'n_estimators': 5000,
                   'learning_rate': 0.01, 'min_child_samples': 20,
                   'subsample': 0.8, 'colsample_bytree': 0.8,
                   'reg_alpha': 0.1, 'reg_lambda': 0.1},
    },
    'Insurer_H_price': {
        'drop_features':  ['old_vehicle_flag',
                           'postal_code_social_benefit_recipients_ratio'],
        'maker_encode':   True,
        'log_transform':  True,
        'params': {'num_leaves': 63, 'n_estimators': 5000,
                   'learning_rate': 0.01, 'min_child_samples': 30,
                   'subsample': 0.8, 'colsample_bytree': 0.8,
                   'reg_alpha': 0.2, 'reg_lambda': 0.2},
    },
    'Insurer_I_price': {
        'drop_features':  ['vehicle_power', 'vehicle_gross_weight',
                           'vehicle_net_weight', 'vehicle_length',
                           'vehicle_width', 'vehicle_number_of_doors'],
        'maker_encode':   False,
        'log_transform':  True,
        'params': {'num_leaves': 127, 'n_estimators': 5000,
                   'learning_rate': 0.01, 'min_child_samples': 20,
                   'subsample': 0.8, 'colsample_bytree': 0.8,
                   'reg_alpha': 0.1, 'reg_lambda': 0.1},
    },
    'Insurer_J_price': {
        'drop_features':  ['vehicle_length', 'vehicle_width',
                           'vehicle_number_of_doors', 'old_vehicle_flag'],
        'maker_encode':   False,
        'log_transform':  True,
        'params': {'num_leaves': 127, 'n_estimators': 5000,
                   'learning_rate': 0.01, 'min_child_samples': 20,
                   'subsample': 0.8, 'colsample_bytree': 0.8,
                   'reg_alpha': 0.1, 'reg_lambda': 0.1},
    },
    'Insurer_K_price': {
        'drop_features':  ['vehicle_length', 'vehicle_width',
                           'vehicle_number_of_doors', 'old_vehicle_flag'],
        'maker_encode':   False,
        'log_transform':  True,
        'params': {'num_leaves': 127, 'n_estimators': 5000,
                   'learning_rate': 0.01, 'min_child_samples': 20,
                   'subsample': 0.8, 'colsample_bytree': 0.8,
                   'reg_alpha': 0.1, 'reg_lambda': 0.1},
    },
}

# Training loop
models = {}
preds  = {}
scores = {}

print("=== TRAINING 11 INSURER MODELS ===\n")

for insurer, cfg in INSURER_CONFIG.items():
    # Build feature set for this insurer
    drop   = cfg.get('drop_features', [])
    f_cols = [c for c in base_feature_cols if c not in drop]

    # Add per-insurer maker encoding if needed
    if cfg.get('maker_encode', False):
        enc         = maker_encodings[insurer]
        global_mean = np.mean(list(enc.values()))
        col_name    = f'maker_enc_{insurer}'
        df_train[col_name] = df_train_raw['vehicle_maker'].map(enc).fillna(global_mean)
        df_test[col_name]  = df_test_raw['vehicle_maker'].map(enc).fillna(global_mean)
        f_cols.append(col_name)

    # Get training data
    mask = df_train[insurer].notnull()
    X    = df_train.loc[mask, f_cols]
    y    = df_train.loc[mask, insurer]

    use_log = cfg.get('log_transform', False)
    y_fit   = np.log1p(y) if use_log else y

    X_train, X_val, y_train, y_val = train_test_split(
        X, y_fit, test_size=0.1, random_state=RANDOM_STATE
    )

    params = {
        'objective': 'regression_l1',
        'metric': 'mae',
        'random_state': RANDOM_STATE,
        'verbose': -1,
        **cfg['params']
    }

    model = lgb.LGBMRegressor(**params)
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(100, verbose=False),
                   lgb.log_evaluation(period=-1)]
    )

    val_pred = model.predict(X_val)
    if use_log:
        val_pred = np.expm1(val_pred)
        y_val    = np.expm1(y_val)
    mae = mean_absolute_error(y_val, val_pred)

    test_pred = model.predict(df_test[f_cols])
    if use_log:
        test_pred = np.expm1(test_pred)

    scores[insurer] = mae
    preds[insurer]  = test_pred
    models[insurer] = model

    # Save model
    with open(f'saved_models_v2/{insurer}.pkl', 'wb') as f:
        pickle.dump(model, f)

    print(f"  {insurer}: MAE={mae:.2f}  n={mask.sum():,}  "
          f"features={len(f_cols)}  best_iter={model.best_iteration_}")

# Weighted MAE
total_errors = sum(scores[i] * df_train[i].notnull().sum()
                   for i in INSURER_CONFIG)
total_rows   = sum(df_train[i].notnull().sum() for i in INSURER_CONFIG)
print(f"\nWeighted pooled MAE: {total_errors/total_rows:.4f}")

=== TRAINING 11 INSURER MODELS ===

  Insurer_A_price: MAE=6.05  n=527,372  features=111  best_iter=5000
  Insurer_B_price: MAE=6.05  n=435,759  features=107  best_iter=5000
  Insurer_C_price: MAE=4.72  n=431,011  features=107  best_iter=5000
  Insurer_D_price: MAE=6.16  n=403,079  features=108  best_iter=5000
  Insurer_E_price: MAE=14.67  n=402,556  features=109  best_iter=5000
  Insurer_F_price: MAE=7.75  n=357,844  features=107  best_iter=5000
  Insurer_G_price: MAE=4.47  n=314,221  features=107  best_iter=5000
  Insurer_H_price: MAE=6.73  n=113,403  features=110  best_iter=5000
  Insurer_I_price: MAE=11.01  n=289,429  features=105  best_iter=5000
  Insurer_J_price: MAE=5.47  n=306,099  features=107  best_iter=5000
  Insurer_K_price: MAE=8.59  n=357,687  features=107  best_iter=5000

Weighted pooled MAE: 7.3945


In [ ]:
import pickle
import os
import pandas as pd
import numpy as np

# ── Load saved models ─────────────────────────────────────────────────────
insurer_cols = [
    'Insurer_A_price', 'Insurer_B_price', 'Insurer_C_price',
    'Insurer_D_price', 'Insurer_E_price', 'Insurer_F_price',
    'Insurer_G_price', 'Insurer_H_price', 'Insurer_I_price',
    'Insurer_J_price', 'Insurer_K_price'
]

models = {}
for insurer in insurer_cols:
    path = f'saved_models_v2/{insurer}.pkl'
    if os.path.exists(path):
        with open(path, 'rb') as f:
            models[insurer] = pickle.load(f)
        print(f"Loaded: {insurer}")
    else:
        print(f"MISSING: {insurer}")

Loaded: Insurer_A_price
Loaded: Insurer_B_price
Loaded: Insurer_C_price
Loaded: Insurer_D_price
Loaded: Insurer_E_price
Loaded: Insurer_F_price
Loaded: Insurer_G_price
Loaded: Insurer_H_price
Loaded: Insurer_I_price
Loaded: Insurer_J_price
Loaded: Insurer_K_price


In [15]:
print("\nFiles in saved_models_v2:")
if os.path.exists('saved_models_v2'):
    for f in sorted(os.listdir('saved_models_v2')):
        size_kb = os.path.getsize(f'saved_models_v2/{f}') / 1024
        print(f"  {f:50s} {size_kb:.0f} KB")
else:
    print("Directory not found!")


Files in saved_models_v2:
  Insurer_A_price.pkl                                59451 KB
  Insurer_B_price.pkl                                59239 KB
  Insurer_C_price.pkl                                59312 KB
  Insurer_D_price.pkl                                59180 KB
  Insurer_E_price.pkl                                29855 KB
  Insurer_F_price.pkl                                58935 KB
  Insurer_G_price.pkl                                29746 KB
  Insurer_H_price.pkl                                29215 KB
  Insurer_I_price.pkl                                58473 KB
  Insurer_J_price.pkl                                59105 KB
  Insurer_K_price.pkl                                58713 KB


In [16]:
# ── Build feature sets and predict ───────────────────────────────────────
base_feature_cols = [c for c in df_train.columns
                     if not c.endswith('_price')
                     and not c.endswith('_quoted')
                     and c != 'coverage']

INSURER_CONFIG = {
    'Insurer_A_price': {
        'drop_features': ['postal_code_social_benefit_recipients_ratio'],
        'maker_encode':  True,
    },
    'Insurer_B_price': {
        'drop_features': ['vehicle_length', 'vehicle_width',
                          'vehicle_number_of_doors', 'old_vehicle_flag'],
        'maker_encode':  False,
    },
    'Insurer_C_price': {
        'drop_features': ['vehicle_length', 'vehicle_width',
                          'vehicle_number_of_doors', 'old_vehicle_flag'],
        'maker_encode':  False,
    },
    'Insurer_D_price': {
        'drop_features': ['vehicle_length', 'vehicle_width',
                          'old_vehicle_flag'],
        'maker_encode':  False,
    },
    'Insurer_E_price': {
        'drop_features': ['vehicle_length', 'vehicle_width',
                          'vehicle_number_of_doors'],
        'maker_encode':  True,
    },
    'Insurer_F_price': {
        'drop_features': ['vehicle_length', 'vehicle_width',
                          'vehicle_number_of_doors', 'old_vehicle_flag'],
        'maker_encode':  False,
    },
    'Insurer_G_price': {
        'drop_features': ['vehicle_length', 'vehicle_width',
                          'vehicle_number_of_doors', 'old_vehicle_flag'],
        'maker_encode':  False,
    },
    'Insurer_H_price': {
        'drop_features': ['old_vehicle_flag',
                          'postal_code_social_benefit_recipients_ratio'],
        'maker_encode':  True,
    },
    'Insurer_I_price': {
        'drop_features': ['vehicle_power', 'vehicle_gross_weight',
                          'vehicle_net_weight', 'vehicle_length',
                          'vehicle_width', 'vehicle_number_of_doors'],
        'maker_encode':  False,
    },
    'Insurer_J_price': {
        'drop_features': ['vehicle_length', 'vehicle_width',
                          'vehicle_number_of_doors', 'old_vehicle_flag'],
        'maker_encode':  False,
    },
    'Insurer_K_price': {
        'drop_features': ['vehicle_length', 'vehicle_width',
                          'vehicle_number_of_doors', 'old_vehicle_flag'],
        'maker_encode':  False,
    },
}

preds = {}

print("=== GENERATING PREDICTIONS ===\n")
for insurer, cfg in INSURER_CONFIG.items():
    drop   = cfg.get('drop_features', [])
    f_cols = [c for c in base_feature_cols if c not in drop]

    if cfg.get('maker_encode', False):
        enc         = maker_encodings[insurer]
        global_mean = np.mean(list(enc.values()))
        col_name    = f'maker_enc_{insurer}'
        df_test[col_name] = df_test_raw['vehicle_maker'].map(enc).fillna(global_mean)
        f_cols.append(col_name)

    pred = np.expm1(models[insurer].predict(df_test[f_cols]))
    pred = np.clip(pred, 0, None)
    preds[insurer] = pred
    print(f"  {insurer}: mean={pred.mean():.2f}  "
          f"min={pred.min():.2f}  max={pred.max():.2f}")

=== GENERATING PREDICTIONS ===

  Insurer_A_price: mean=92.76  min=14.01  max=3371.69
  Insurer_B_price: mean=105.52  min=17.18  max=805.79
  Insurer_C_price: mean=98.53  min=13.88  max=677.17
  Insurer_D_price: mean=103.83  min=15.65  max=637.80
  Insurer_E_price: mean=116.47  min=19.65  max=1318.71
  Insurer_F_price: mean=90.02  min=15.37  max=698.24
  Insurer_G_price: mean=99.81  min=14.87  max=564.97
  Insurer_H_price: mean=126.81  min=20.53  max=572.73
  Insurer_I_price: mean=108.65  min=18.17  max=673.32
  Insurer_J_price: mean=124.44  min=22.79  max=885.81
  Insurer_K_price: mean=126.96  min=19.62  max=704.37


In [17]:
# ── Build submission ──────────────────────────────────────────────────────
insurer_cols_formatted = [
    'Insurer A price', 'Insurer B price', 'Insurer C price',
    'Insurer D price', 'Insurer E price', 'Insurer F price',
    'Insurer G price', 'Insurer H price', 'Insurer I price',
    'Insurer J price', 'Insurer K price'
]

quote_ids  = df_test_raw['quote_id'].values
submission = pd.DataFrame({'quote_id': quote_ids})
for col in insurer_cols_formatted:
    submission[col] = 0.0

for insurer, formatted in zip(insurer_cols, insurer_cols_formatted):
    if insurer in preds:
        submission[formatted] = preds[insurer]

submission[insurer_cols_formatted] = (
    submission[insurer_cols_formatted].round(2).clip(lower=0)
)

submission.to_csv('submission_v5.csv', sep=';', decimal='.', index=False)

import os
size_mb = os.path.getsize('submission_v5.csv') / (1024*1024)
print(f"\nSaved: submission_v5.csv  ({size_mb:.1f} MB)")
print(f"Shape: {submission.shape}")
print(f"Nulls: {submission.isnull().sum().sum()}")
print(submission.head(3).to_string())


Saved: submission_v5.csv  (11.9 MB)
Shape: (164092, 12)
Nulls: 0
   quote_id  Insurer A price  Insurer B price  Insurer C price  Insurer D price  Insurer E price  Insurer F price  Insurer G price  Insurer H price  Insurer I price  Insurer J price  Insurer K price
0    541293            64.40            80.69            63.69           114.06            98.99            61.94            62.23            89.33           141.99           180.21           144.06
1    541294            28.62            30.56            33.70            34.56            42.57            38.39            28.03            69.07            29.50            42.60            47.15
2    541295            27.85            36.77            32.94            32.74            43.14            32.53            35.66            35.37            32.52            45.49            35.52


In [18]:
# Deep dive into Insurer E
e_data = df_train_raw[df_train_raw['Insurer_E_price'].notnull()].copy()
e_data['claim_free_years_num'] = pd.to_numeric(e_data['claim_free_years'], errors='coerce')
e_data['contractor_age'] = (
    pd.Timestamp('2025-01-01') - 
    pd.to_datetime(e_data['contractor_birthdate'], dayfirst=True, errors='coerce')
).dt.days // 365
e_data['vehicle_value_new_num'] = pd.to_numeric(e_data['vehicle_value_new'], errors='coerce')

# 1. Price distribution
print("=== INSURER E PRICE DISTRIBUTION ===")
print(e_data['Insurer_E_price'].describe().round(2))
print(f"\nSkew: {e_data['Insurer_E_price'].skew():.2f}")

# 2. What % of prices are above 500?
high_price = (e_data['Insurer_E_price'] > 500).sum()
print(f"\nPrices > 500: {high_price:,} ({high_price/len(e_data)*100:.2f}%)")
print(f"Prices > 300: {(e_data['Insurer_E_price'] > 300).sum():,}")
print(f"Prices > 200: {(e_data['Insurer_E_price'] > 200).sum():,}")

# 3. What drives high E prices?
print("\n=== WHAT DRIVES HIGH E PRICES ===")
e_data['price_segment'] = pd.cut(
    e_data['Insurer_E_price'],
    bins=[0, 100, 200, 500, 9999],
    labels=['low', 'medium', 'high', 'very_high']
)

print(e_data.groupby('price_segment').agg(
    count=('Insurer_E_price', 'count'),
    mean_price=('Insurer_E_price', 'mean'),
    mean_age=('contractor_age', 'mean'),
    mean_cfr=('claim_free_years_num', 'mean'),
    mean_value=('vehicle_value_new_num', 'mean'),
).round(2).to_string())

# 4. Coverage breakdown for high prices
print("\n=== HIGH E PRICES BY COVERAGE ===")
print(e_data[e_data['Insurer_E_price'] > 300].groupby('coverage')['Insurer_E_price']
      .agg(['count', 'mean', 'min', 'max']).round(2))

# 5. Vehicle maker for very high E prices
print("\n=== TOP MAKERS FOR VERY HIGH E PRICES (>500) ===")
print(e_data[e_data['Insurer_E_price'] > 500]['vehicle_maker']
      .value_counts().head(15))

=== INSURER E PRICE DISTRIBUTION ===
count    402556.00
mean        115.72
std          88.18
min          15.55
25%          61.56
50%          90.29
75%         139.66
max        2283.81
Name: Insurer_E_price, dtype: float64

Skew: 3.26

Prices > 500: 2,950 (0.73%)
Prices > 300: 16,445
Prices > 200: 46,222

=== WHAT DRIVES HIGH E PRICES ===
                count  mean_price  mean_age  mean_cfr  mean_value
price_segment                                                    
low            227854       65.08     40.25      8.54    27386.14
medium         128480      138.05     32.62      3.38    32892.40
high            43272      279.64     24.95      1.18    33654.40
very_high        2950      650.94     20.95      0.28    34075.92

=== HIGH E PRICES BY COVERAGE ===
               count    mean     min      max
coverage                                     
casco           4627  391.22  300.09  1074.13
limited_casco   5911  443.36  300.02  2234.69
mtpl            5907  414.73  300.03  22

In [19]:
# Check age vs price relationship for E
print("=== INSURER E: AGE BUCKETS VS PRICE ===")
e_data['age_bucket'] = pd.cut(
    e_data['contractor_age'],
    bins=[0, 18, 21, 25, 30, 35, 40, 50, 60, 100],
    labels=['<18', '18-21', '21-25', '25-30', 
            '30-35', '35-40', '40-50', '50-60', '60+']
)
print(e_data.groupby('age_bucket')['Insurer_E_price']
      .agg(['mean', 'median', 'count']).round(2).to_string())

# Check CFR vs price
print("\n=== INSURER E: CLAIM FREE YEARS VS PRICE ===")
cfr_price = e_data.groupby('claim_free_years_num')['Insurer_E_price']\
    .agg(['mean', 'median', 'count']).round(2)
print(cfr_price[cfr_price['count'] > 100].sort_index().to_string())

# Check interaction — young + zero CFR
print("\n=== YOUNG (< 25) + ZERO CFR ===")
young_zero = e_data[
    (e_data['contractor_age'] < 25) & 
    (e_data['claim_free_years_num'] == 0)
]['Insurer_E_price']
print(f"Count: {len(young_zero):,}")
print(f"Mean:  {young_zero.mean():.2f}")
print(f"Median:{young_zero.median():.2f}")
print(f"Std:   {young_zero.std():.2f}")

=== INSURER E: AGE BUCKETS VS PRICE ===
              mean  median  count
age_bucket                       
<18         299.99  257.52  21722
18-21       184.32  167.13  40398
21-25       118.96  102.89  59803
25-30        96.58   80.96  65269
30-35        90.95   75.76  54705
35-40        93.88   79.29  33970
40-50        94.25   81.73  50059
50-60        84.14   70.73  40275
60+          80.68   67.35  36355

=== INSURER E: CLAIM FREE YEARS VS PRICE ===
                        mean  median   count
claim_free_years_num                        
-2                    153.91  134.73    1728
-1                    149.24  125.76    1955
 0                    173.33  141.74  116024
 1                    135.43  113.24   35593
 2                    113.74   95.78   33653
 3                    101.30   84.74   26712
 4                     92.76   79.72   20345
 5                     88.21   73.95   25497
 6                     80.81   67.85   15408
 7                     77.86   64.92   12571


In [20]:
# Add E-specific features to df_train and df_test
for df in [df_train, df_test]:
    # Age risk score — mirrors E's pricing curve
    df['e_age_risk'] = pd.cut(
        df['contractor_age'],
        bins=[0, 18, 21, 25, 30, 35, 100],
        labels=[5, 4, 3, 2, 1, 0]
    ).astype(float)
    
    # CFR risk — non-linear, biggest drop in first 5 years
    df['e_cfr_risk'] = np.where(df['claim_free_years'] < 0, 3,
                       np.where(df['claim_free_years'] == 0, 2,
                       np.where(df['claim_free_years'] <= 3, 1, 0)))
    
    # Key interaction — young AND inexperienced
    df['e_young_inexperienced'] = (
        (df['contractor_age'] < 25) & 
        (df['claim_free_years'] <= 1)
    ).astype(int)
    
    # Age × CFR interaction
    df['e_age_x_cfr'] = df['contractor_age'] * df['claim_free_years'].clip(lower=0)
    
    # Inverse age — younger = higher value
    df['e_inv_age'] = 1 / (df['contractor_age'] + 1)
    
    # Inverse CFR — less experience = higher value  
    df['e_inv_cfr'] = 1 / (df['claim_free_years'].clip(lower=0) + 1)
    
    # Combined risk score
    df['e_risk_score'] = df['e_inv_age'] * df['e_inv_cfr']

print("E-specific features added")
print(df_train[['e_age_risk', 'e_cfr_risk', 'e_young_inexperienced',
                'e_age_x_cfr', 'e_inv_age', 'e_inv_cfr', 
                'e_risk_score']].describe().round(3))

E-specific features added
       e_age_risk  e_cfr_risk  e_young_inexperienced  e_age_x_cfr   e_inv_age  \
count  541288.000  541292.000             541292.000   541292.000  541292.000   
mean        1.455       0.854                  0.191      313.989       0.030   
std         1.557       0.884                  0.393      521.786       0.011   
min         0.000       0.000                  0.000        0.000       0.010   
25%         0.000       0.000                  0.000        0.000       0.021   
50%         1.000       1.000                  0.000       90.000       0.029   
75%         3.000       2.000                  0.000      370.000       0.038   
max         5.000       3.000                  1.000     8282.000       0.056   

        e_inv_cfr  e_risk_score  
count  541292.000    541292.000  
mean        0.441         0.016  
std         0.392         0.016  
min         0.012         0.000  
25%         0.100         0.002  
50%         0.250         0.008  
75%   

In [21]:
# Retrain E only with new features
insurer = 'Insurer_E_price'
cfg     = INSURER_CONFIG[insurer]

drop   = cfg.get('drop_features', [])
f_cols = [c for c in base_feature_cols if c not in drop]

# Add E-specific features
e_specific = ['e_age_risk', 'e_cfr_risk', 'e_young_inexperienced',
              'e_age_x_cfr', 'e_inv_age', 'e_inv_cfr', 'e_risk_score']
f_cols = f_cols + e_specific

# Add maker encoding
enc         = maker_encodings[insurer]
global_mean = np.mean(list(enc.values()))
col_name    = f'maker_enc_{insurer}'
df_train[col_name] = df_train_raw['vehicle_maker'].map(enc).fillna(global_mean)
df_test[col_name]  = df_test_raw['vehicle_maker'].map(enc).fillna(global_mean)
f_cols.append(col_name)

mask = df_train[insurer].notnull()
X    = df_train.loc[mask, f_cols]
y    = df_train.loc[mask, insurer]

y_log = np.log1p(y)

X_train, X_val, y_train, y_val = train_test_split(
    X, y_log, test_size=0.1, random_state=RANDOM_STATE
)

params = {
    'objective': 'regression_l1',
    'metric': 'mae',
    'n_estimators': 5000,
    'learning_rate': 0.01,
    'num_leaves': 63,
    'min_child_samples': 50,
    'subsample': 0.7,
    'colsample_bytree': 0.7,
    'reg_alpha': 0.3,
    'reg_lambda': 0.3,
    'random_state': RANDOM_STATE,
    'verbose': -1,
}

model_e = lgb.LGBMRegressor(**params)
model_e.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(100, verbose=False),
               lgb.log_evaluation(period=-1)]
)

val_pred = np.expm1(model_e.predict(X_val))
val_actual = np.expm1(y_val)
mae_e = mean_absolute_error(val_actual, val_pred)

print(f"Insurer E new MAE: {mae_e:.2f}  "
      f"(was 14.67)  best_iter={model_e.best_iteration_}")

# Update predictions
preds['Insurer_E_price'] = np.expm1(
    model_e.predict(df_test[f_cols])
).clip(min=0)

Insurer E new MAE: 14.63  (was 14.67)  best_iter=5000


In [22]:
# Create sample weights for E — upweight young/high-risk rows
mask_e = df_train['Insurer_E_price'].notnull()
X_e    = df_train.loc[mask_e, f_cols]
y_e    = df_train.loc[mask_e, 'Insurer_E_price']

# Build weights — higher weight for harder-to-predict rows
weights = np.ones(len(X_e))

# Upweight young drivers
young_mask = df_train.loc[mask_e, 'contractor_age'] < 25
weights[young_mask.values] = 3.0

# Upweight zero CFR
zero_cfr_mask = df_train.loc[mask_e, 'claim_free_years'] <= 0
weights[zero_cfr_mask.values] *= 2.0

# Upweight very high prices (> 300) — model needs to learn these
high_price_mask = y_e > 300
weights[high_price_mask.values] *= 3.0

print(f"Weight distribution:")
print(f"  Normal rows (weight=1):  {(weights==1).sum():,}")
print(f"  Young only (weight=3):   {((weights==3) & ~zero_cfr_mask.values).sum():,}")
print(f"  Zero CFR only (w=2):     {((weights==2)).sum():,}")
print(f"  Young+ZeroCFR (w=6):     {((weights==6)).sum():,}")
print(f"  High price (w*3):        {high_price_mask.sum():,}")
print(f"  Max weight:              {weights.max():.1f}")

y_log = np.log1p(y_e)

X_train, X_val, y_train, y_val, w_train, w_val = train_test_split(
    X_e, y_log, weights, test_size=0.1, random_state=RANDOM_STATE
)

params = {
    'objective': 'regression_l1',
    'metric': 'mae',
    'n_estimators': 10000,
    'learning_rate': 0.01,
    'num_leaves': 63,
    'min_child_samples': 20,
    'subsample': 0.7,
    'colsample_bytree': 0.7,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'random_state': RANDOM_STATE,
    'verbose': -1,
}

model_e_weighted = lgb.LGBMRegressor(**params)
model_e_weighted.fit(
    X_train, y_train,
    sample_weight=w_train,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(100, verbose=False),
               lgb.log_evaluation(period=-1)]
)

val_pred   = np.expm1(model_e_weighted.predict(X_val))
val_actual = np.expm1(y_val)
mae_weighted = mean_absolute_error(val_actual, val_pred)

print(f"\nInsurer E weighted MAE: {mae_weighted:.2f}  "
      f"(was 14.63)  best_iter={model_e_weighted.best_iteration_}")

# Also check MAE specifically on young drivers
young_val_mask = df_train.loc[
    X_val.index, 'contractor_age'] < 25
if young_val_mask.sum() > 0:
    mae_young = mean_absolute_error(
        val_actual[young_val_mask.values],
        val_pred[young_val_mask.values]
    )
    print(f"MAE on young drivers only: {mae_young:.2f}")

Weight distribution:
  Normal rows (weight=1):  245,072
  Young only (weight=3):   35,983
  Zero CFR only (w=2):     62,365
  Young+ZeroCFR (w=6):     46,593
  High price (w*3):        16,445
  Max weight:              18.0

Insurer E weighted MAE: 15.16  (was 14.63)  best_iter=10000
MAE on young drivers only: 21.06


In [23]:
# Add polynomial age/CFR features
for df in [df_train, df_test]:
    # Age curve features
    df['age_sq']         = df['contractor_age'] ** 2
    df['age_cb']         = df['contractor_age'] ** 3
    df['inv_age']        = 1 / (df['contractor_age'] + 1)
    df['inv_age_sq']     = 1 / (df['contractor_age'] + 1) ** 2

    # CFR curve features  
    df['cfr_clipped']    = df['claim_free_years'].clip(lower=0)
    df['cfr_log']        = np.log1p(df['cfr_clipped'])
    df['inv_cfr']        = 1 / (df['cfr_clipped'] + 1)
    df['inv_cfr_sq']     = 1 / (df['cfr_clipped'] + 1) ** 2

    # Interaction terms
    df['age_x_inv_cfr']  = df['contractor_age'] * df['inv_cfr']
    df['inv_age_x_cfr']  = df['inv_age'] * df['cfr_clipped']
    df['inv_age_x_inv_cfr'] = df['inv_age'] * df['inv_cfr']

poly_features = [
    'age_sq', 'age_cb', 'inv_age', 'inv_age_sq',
    'cfr_log', 'inv_cfr', 'inv_cfr_sq',
    'age_x_inv_cfr', 'inv_age_x_cfr', 'inv_age_x_inv_cfr'
]

print("Polynomial features added")
print(df_train[poly_features].describe().round(3))

Polynomial features added
           age_sq       age_cb     inv_age  inv_age_sq     cfr_log  \
count  541292.000   541292.000  541292.000  541292.000  541292.000   
mean     1626.478    82103.434       0.030       0.001       1.361   
std      1410.620   108659.004       0.011       0.001       1.141   
min       289.000     4913.000       0.010       0.000       0.000   
25%       625.000    15625.000       0.021       0.000       0.000   
50%      1089.000    35937.000       0.029       0.001       1.386   
75%      2209.000   103823.000       0.038       0.001       2.303   
max     10201.000  1030301.000       0.056       0.003       4.419   

          inv_cfr  inv_cfr_sq  age_x_inv_cfr  inv_age_x_cfr  inv_age_x_inv_cfr  
count  541292.000  541292.000     541292.000     541292.000         541292.000  
mean        0.441       0.348         13.540          0.136              0.016  
std         0.392       0.438         12.967          0.150              0.016  
min         0.012  

In [24]:
insurer  = 'Insurer_E_price'
cfg      = INSURER_CONFIG[insurer]
drop     = cfg.get('drop_features', [])
f_cols_e = [c for c in base_feature_cols if c not in drop]
f_cols_e = f_cols_e + poly_features

# Add maker encoding
enc         = maker_encodings[insurer]
global_mean = np.mean(list(enc.values()))
col_name    = f'maker_enc_{insurer}'
df_train[col_name] = df_train_raw['vehicle_maker'].map(enc).fillna(global_mean)
df_test[col_name]  = df_test_raw['vehicle_maker'].map(enc).fillna(global_mean)
f_cols_e.append(col_name)

mask = df_train[insurer].notnull()
X    = df_train.loc[mask, f_cols_e]
y    = df_train.loc[mask, insurer]

X_train, X_val, y_train, y_val = train_test_split(
    X, np.log1p(y), test_size=0.1, random_state=RANDOM_STATE
)

params = {
    'objective': 'regression_l1',
    'metric': 'mae',
    'n_estimators': 2000,
    'learning_rate': 0.02,
    'num_leaves': 63,
    'min_child_samples': 20,
    'subsample': 0.7,
    'colsample_bytree': 0.7,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'random_state': RANDOM_STATE,
    'verbose': -1,
}

model_e_poly = lgb.LGBMRegressor(**params)
model_e_poly.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(150, verbose=False),
               lgb.log_evaluation(period=-1)]
)

val_pred   = np.expm1(model_e_poly.predict(X_val))
val_actual = np.expm1(y_val)
mae_poly   = mean_absolute_error(val_actual, val_pred)

print(f"Insurer E poly MAE: {mae_poly:.2f}  "
      f"(was 14.63)  best_iter={model_e_poly.best_iteration_}")

# Check young drivers specifically
young_mask = df_train.loc[X_val.index, 'contractor_age'] < 25
if young_mask.sum() > 0:
    mae_young = mean_absolute_error(
        val_actual[young_mask.values],
        val_pred[young_mask.values]
    )
    print(f"MAE young drivers: {mae_young:.2f}")

Insurer E poly MAE: 15.01  (was 14.63)  best_iter=2000
MAE young drivers: 25.73


In [25]:
# Deep dive Insurer I
i_data = df_train_raw[df_train_raw['Insurer_I_price'].notnull()].copy()
i_data['claim_free_years_num'] = pd.to_numeric(i_data['claim_free_years'], errors='coerce')
i_data['contractor_age'] = (
    pd.Timestamp('2025-01-01') - 
    pd.to_datetime(i_data['contractor_birthdate'], dayfirst=True, errors='coerce')
).dt.days // 365
i_data['vehicle_age_num'] = pd.to_numeric(i_data['vehicle_age'], errors='coerce')
i_data['vehicle_value_num'] = pd.to_numeric(i_data['vehicle_value_new'], errors='coerce')

print("=== INSURER I PRICE DISTRIBUTION ===")
print(i_data['Insurer_I_price'].describe().round(2))
print(f"Skew: {i_data['Insurer_I_price'].skew():.2f}")

print("\n=== I PRICE BY COVERAGE ===")
print(i_data.groupby('coverage')['Insurer_I_price']
      .agg(['mean', 'median', 'std', 'count']).round(2))

print("\n=== I PRICE BY AGE BUCKET ===")
i_data['age_bucket'] = pd.cut(
    i_data['contractor_age'],
    bins=[0, 25, 35, 45, 55, 100],
    labels=['<25', '25-35', '35-45', '45-55', '55+']
)
print(i_data.groupby('age_bucket')['Insurer_I_price']
      .agg(['mean', 'median', 'count']).round(2))

print("\n=== I PRICE BY VEHICLE AGE ===")
i_data['veh_age_bucket'] = pd.cut(
    i_data['vehicle_age_num'],
    bins=[0, 3, 7, 12, 20, 100],
    labels=['new', 'recent', 'mid', 'old', 'very_old']
)
print(i_data.groupby('veh_age_bucket')['Insurer_I_price']
      .agg(['mean', 'median', 'count']).round(2))

print("\n=== I QUOTE RATE BY COVERAGE ===")
for cov in ['mtpl', 'limited_casco', 'casco']:
    total = len(df_train_raw[df_train_raw['coverage'] == cov])
    quoted = i_data[i_data['coverage'] == cov].shape[0]
    print(f"  {cov:15s} {quoted:,} / {total:,} = {quoted/total*100:.1f}%")

=== INSURER I PRICE DISTRIBUTION ===
count    289429.00
mean         94.85
std          71.10
min          14.05
25%          49.38
50%          70.01
75%         113.84
max         901.98
Name: Insurer_I_price, dtype: float64
Skew: 2.29

=== I PRICE BY COVERAGE ===
                 mean  median    std   count
coverage                                    
casco           97.84   75.25  62.52   86186
limited_casco   85.18   61.21  70.03  154673
mtpl           120.36   96.44  81.29   48570

=== I PRICE BY AGE BUCKET ===
              mean  median  count
age_bucket                       
<25         155.01  130.01  57582
25-35        84.84   65.46  83882
35-45        82.13   64.61  58511
45-55        74.85   59.62  42536
55+          72.91   59.46  46918

=== I PRICE BY VEHICLE AGE ===
                  mean  median  count
veh_age_bucket                       
new             109.18   86.61  24836
recent           89.28   67.43  77899
mid              79.47   59.52  83774
old             1

In [26]:
# Add I-specific features
for df in [df_train, df_test]:
    # U-shaped vehicle age signal
    df['i_veh_age_risk'] = np.where(
        df['vehicle_age'] <= 3,  2,   # new = risky
        np.where(df['vehicle_age'] <= 7, 0,   # recent = cheapest
        np.where(df['vehicle_age'] <= 12, 1,  # mid = moderate
        2)))                                   # old = risky again

    # mtpl flag — I prices mtpl highest
    df['i_is_mtpl'] = (df['coverage'] == 0).astype(int)

    # Young + new car combination
    df['i_young_new_car'] = (
        (df['contractor_age'] < 25) &
        (df['vehicle_age'] <= 3)
    ).astype(int)

i_specific = ['i_veh_age_risk', 'i_is_mtpl', 'i_young_new_car']

# Retrain I with specific features
insurer  = 'Insurer_I_price'
cfg      = INSURER_CONFIG[insurer]
drop     = cfg.get('drop_features', [])
f_cols_i = [c for c in base_feature_cols if c not in drop] + i_specific

mask = df_train[insurer].notnull()
X    = df_train.loc[mask, f_cols_i]
y    = df_train.loc[mask, insurer]

X_train, X_val, y_train, y_val = train_test_split(
    X, np.log1p(y), test_size=0.1, random_state=RANDOM_STATE
)

params = {
    'objective': 'regression_l1',
    'metric': 'mae',
    'n_estimators': 2000,
    'learning_rate': 0.02,
    'num_leaves': 127,
    'min_child_samples': 20,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'random_state': RANDOM_STATE,
    'verbose': -1,
}

model_i = lgb.LGBMRegressor(**params)
model_i.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(100, verbose=False),
               lgb.log_evaluation(period=-1)]
)

val_pred   = np.expm1(model_i.predict(X_val))
val_actual = np.expm1(y_val)
mae_i      = mean_absolute_error(val_actual, val_pred)

print(f"Insurer I new MAE: {mae_i:.2f}  "
      f"(was 11.01)  best_iter={model_i.best_iteration_}")

if mae_i < 11.01:
    preds['Insurer_I_price'] = np.expm1(
        model_i.predict(df_test[f_cols_i])
    ).clip(min=0)
    print("IMPROVED — predictions updated")
else:
    print("NO IMPROVEMENT — keeping original")

Insurer I new MAE: 11.34  (was 11.01)  best_iter=2000
NO IMPROVEMENT — keeping original


In [27]:
from xgboost import XGBRegressor

insurer  = 'Insurer_E_price'
cfg      = INSURER_CONFIG[insurer]
drop     = cfg.get('drop_features', [])
f_cols_e = [c for c in base_feature_cols if c not in drop]

# Add maker encoding
enc         = maker_encodings[insurer]
global_mean = np.mean(list(enc.values()))
col_name    = 'maker_enc_Insurer_E_price'
df_train[col_name] = df_train_raw['vehicle_maker'].map(enc).fillna(global_mean)
df_test[col_name]  = df_test_raw['vehicle_maker'].map(enc).fillna(global_mean)
f_cols_e.append(col_name)

mask = df_train[insurer].notnull()
X    = df_train.loc[mask, f_cols_e]
y    = df_train.loc[mask, insurer]

X_train, X_val, y_train, y_val = train_test_split(
    X, np.log1p(y), test_size=0.1, random_state=RANDOM_STATE
)

model_xgb = XGBRegressor(
    objective='reg:absoluteerror',
    n_estimators=2000,
    learning_rate=0.02,
    max_depth=7,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=RANDOM_STATE,
    verbosity=0,
    early_stopping_rounds=100,
)

model_xgb.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)

val_pred   = np.expm1(model_xgb.predict(X_val))
val_actual = np.expm1(y_val)
mae_xgb    = mean_absolute_error(val_actual, val_pred)

print(f"XGBoost E MAE: {mae_xgb:.2f}  "
      f"(was 14.63)  best_iter={model_xgb.best_iteration}")

if mae_xgb < 14.63:
    preds['Insurer_E_price'] = np.expm1(
        model_xgb.predict(df_test[f_cols_e])
    ).clip(min=0)
    print("IMPROVED — predictions updated")
else:
    print("NO IMPROVEMENT — keeping original")

XGBoost E MAE: 16.25  (was 14.63)  best_iter=1999
NO IMPROVEMENT — keeping original


In [28]:
from catboost import CatBoostRegressor

insurer  = 'Insurer_E_price'
cfg      = INSURER_CONFIG[insurer]
drop     = cfg.get('drop_features', [])
f_cols_e = [c for c in base_feature_cols if c not in drop]

col_name = 'maker_enc_Insurer_E_price'
if col_name not in df_train.columns:
    enc         = maker_encodings[insurer]
    global_mean = np.mean(list(enc.values()))
    df_train[col_name] = df_train_raw['vehicle_maker'].map(enc).fillna(global_mean)
    df_test[col_name]  = df_test_raw['vehicle_maker'].map(enc).fillna(global_mean)
f_cols_e.append(col_name)

mask = df_train[insurer].notnull()
X    = df_train.loc[mask, f_cols_e]
y    = df_train.loc[mask, insurer]

X_train, X_val, y_train, y_val = train_test_split(
    X, np.log1p(y), test_size=0.1, random_state=RANDOM_STATE
)

model_cat = CatBoostRegressor(
    loss_function='MAE',
    iterations=2000,
    learning_rate=0.02,
    depth=8,
    random_seed=RANDOM_STATE,
    verbose=0,
)
model_cat.fit(
    X_train, y_train,
    eval_set=(X_val, y_val),
    early_stopping_rounds=100,
)

val_pred   = np.expm1(model_cat.predict(X_val))
val_actual = np.expm1(y_val)
mae_cat    = mean_absolute_error(val_actual, val_pred)

print(f"CatBoost E MAE: {mae_cat:.2f}  "
      f"(was 14.63)  best_iter={model_cat.best_iteration_}")

if mae_cat < 14.63:
    preds['Insurer_E_price'] = np.expm1(
        model_cat.predict(df_test[f_cols_e])
    ).clip(min=0)
    print("IMPROVED — predictions updated")
else:
    print("NO IMPROVEMENT — keeping original")

CatBoost E MAE: 16.78  (was 14.63)  best_iter=1999
NO IMPROVEMENT — keeping original


In [ ]:
# Load Block 3
df_block3_raw = pd.read_parquet('../Hackathon2026Task-20260328T120731Z-3-001/Hackathon2026Task/block3_test.parquet')
print(f"Block 3 shape: {df_block3_raw.shape}")
print(f"Columns match Block 2: {set(df_block3_raw.columns) == set(df_test_raw.columns)}")